### 05 - Build CNN Classifier

#### Objective

Build and validate a custom PyTorch Convolutional Neural Network
for product defect image classification.

This notebook focuses on:

- CNN architecture
- convolutional feature extraction
- tensor shape transitions
- model parameters
- forward-pass validation
- multiclass logits

No model training is performed in this notebook.

##### 1. Imports

In [0]:
import random

import numpy as np
import torch
import torch.nn as nn

from src.project_config import (
    BATCH_SIZE,
    CLASSIFIER_HIDDEN_DIM,
    CNN_CHANNELS,
    IMAGE_CHANNELS,
    IMAGE_HEIGHT,
    IMAGE_WIDTH,
    RANDOM_SEED,
)

from src.cnn_model import ProductDefectCNN

##### 2. Reproducibility

In [0]:
random.seed(RANDOM_SEED)

np.random.seed(RANDOM_SEED)

torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

##### Reproducibility

A fixed random seed makes parameter initialization and test
inputs more reproducible during development.

Exact reproducibility can still depend on hardware, library
versions, and specific GPU operations.

##### 3. Runtime device

In [0]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("PyTorch version:", torch.__version__)
print("Device:", device)

##### 4. Architecture configuration

In [0]:
channel_1, channel_2, channel_3 = CNN_CHANNELS

print(
    "Input channels:",
    IMAGE_CHANNELS,
)

print(
    "CNN channels:",
    CNN_CHANNELS,
)

print(
    "Classifier hidden dimension:",
    CLASSIFIER_HIDDEN_DIM,
)

##### Parameters vs Hyperparameters

The architecture values:

- 32 filters
- 64 filters
- 128 filters
- 64 classifier neurons

are hyperparameters selected when designing the network.

The numerical convolution weights, linear-layer weights,
and biases are parameters learned during model training.

##### 5. Define the custom CNN

In [0]:
class ProductDefectCNN(nn.Module):
    """
    Custom convolutional neural network for multiclass
    product defect image classification.
    """

    def __init__(
        self,
        num_classes: int,
    ):
        super().__init__()

        if num_classes < 2:
            raise ValueError(
                "num_classes must be at least 2."
            )

        channel_1, channel_2, channel_3 = CNN_CHANNELS

        self.features = nn.Sequential(
            nn.Conv2d(
                in_channels=IMAGE_CHANNELS,
                out_channels=channel_1,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=2,
                stride=2,
            ),

            nn.Conv2d(
                in_channels=channel_1,
                out_channels=channel_2,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=2,
                stride=2,
            ),

            nn.Conv2d(
                in_channels=channel_2,
                out_channels=channel_3,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=2,
                stride=2,
            ),
        )

        self.global_pool = nn.AdaptiveAvgPool2d(
            output_size=(1, 1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),

            nn.Linear(
                in_features=channel_3,
                out_features=CLASSIFIER_HIDDEN_DIM,
            ),
            nn.ReLU(),

            nn.Linear(
                in_features=CLASSIFIER_HIDDEN_DIM,
                out_features=num_classes,
            ),
        )

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        """
        Run the forward pass and return raw class logits.
        """

        x = self.features(x)

        x = self.global_pool(x)

        logits = self.classifier(x)

        return logits

##### 6. Why separate features and classifier?

##### Architecture Separation

The model contains two main responsibilities.

##### Feature extractor

The convolutional layers learn spatial visual representations:

RGB pixels
- feature maps
- increasingly useful learned visual features

##### Classifier

The classifier converts the learned visual representation into
one logit for each defect class.

This separation will also make later transfer-learning and
Explainable AI concepts easier to understand.


``` text

IMAGE
  ↓
self.features
  ↓
visual features
  ↓
self.global_pool
  ↓
compact feature vector
  ↓
self.classifier
  ↓
class logits

```

##### 7. Instantiate the model

In [0]:
TEST_NUM_CLASSES = 4

model = ProductDefectCNN(
    num_classes=TEST_NUM_CLASSES
).to(device)

print(model)

Later: num_classes = len(train_dataset.classes)

##### 8. Create a synthetic image batch

In [0]:
sample_images = torch.randn(
    BATCH_SIZE,
    IMAGE_CHANNELS,
    IMAGE_HEIGHT,
    IMAGE_WIDTH,
    device=device,
)

print(
    "Sample input shape:",
    sample_images.shape,
)

##### 9. Validate forward pass

In [0]:
model.eval()

with torch.no_grad():
    sample_logits = model(
        sample_images
    )

print(
    "Output logits shape:",
    sample_logits.shape,
)

##### 10. Validate expected output shape

In [0]:
expected_output_shape = (
    BATCH_SIZE,
    TEST_NUM_CLASSES,
)

assert (
    tuple(sample_logits.shape)
    == expected_output_shape
), (
    "Unexpected output shape. "
    f"Expected {expected_output_shape}, "
    f"received {tuple(sample_logits.shape)}."
)

print(
    "Forward-pass shape validation: PASS"
)

##### 11. Trace intermediate tensor shapes

In [0]:
# Instead of modifying forward() with debugging prints, inspect the architecture externally.

x = sample_images

print(
    "Input:",
    tuple(x.shape),
)

with torch.no_grad():

    for layer in model.features:

        x = layer(x)

        print(
            f"{layer.__class__.__name__:20s}",
            tuple(x.shape),
        )

    x = model.global_pool(x)

    print(
        f"{'AdaptiveAvgPool2d':20s}",
        tuple(x.shape),
    )

    x = model.classifier(x)

    print(
        f"{'Classifier output':20s}",
        tuple(x.shape),
    )

##### 12. Inspect trainable parameters

In [0]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(
    f"Total parameters: "
    f"{total_parameters:,}"
)

print(
    f"Trainable parameters: "
    f"{trainable_parameters:,}"
)

##### 13. Inspect convolution parameters separately

In [0]:
for name, module in model.named_modules():

    if isinstance(
        module,
        nn.Conv2d,
    ):
        print(
            name,
            "weight shape:",
            tuple(module.weight.shape),
        )

        print(
            name,
            "bias shape:",
            tuple(module.bias.shape),
        )

##### 14. Verify logits are not probabilities

In [0]:
print(
    "First image logits:",
    sample_logits[0],
)

In [0]:
sample_probabilities = torch.softmax(
    sample_logits,
    dim=1,
)

print(
    "First image probabilities:",
    sample_probabilities[0],
)

print(
    "Probability sum:",
    sample_probabilities[0].sum().item(),
)

##### 15. Architecture summary

The custom CNN follows:

Input RGB image
[3, 224, 224]

→ Conv2d: 3 → 32
→ ReLU
→ MaxPool

→ Conv2d: 32 → 64
→ ReLU
→ MaxPool

→ Conv2d: 64 → 128
→ ReLU
→ MaxPool

→ Adaptive Average Pooling
→ Flatten

→ Linear: 128 → 64
→ ReLU

→ Linear: 64 → number of classes

→ Raw class logits

##### 16. 🔍 Explainable AI note


The final convolutional layers preserve spatial feature maps.

Later, Grad-CAM will use information associated with deeper
convolutional representations to help identify image regions
that strongly contributed to a class prediction.

This can help determine whether the CNN is focusing on the
actual defect or on irrelevant visual cues such as backgrounds,
labels, borders, or other artifacts.

##### 17. ⚖️ Responsible AI note


Successfully executing a forward pass only verifies that the
architecture is technically valid.

It does not demonstrate that the model is accurate, reliable,
fair across relevant conditions, or appropriate for production.

Later evaluation will include:

- per-class metrics
- confusion matrix
- false-positive and false-negative analysis
- confidence analysis
- failure-case inspection
- data-quality considerations
- production-condition representativeness
- human-review considerations

##### 18. Key Learnings


1. A CNN accepts image tensors in PyTorch's
   [batch, channels, height, width] format.

2. Batch size and convolution output channels are independent.

3. Conv2d(3, 32, 3) receives three RGB channels and produces
   32 learned feature maps.

4. The number of output channels equals the number of filters
   in that convolutional layer.

5. The output feature maps from one convolution become the
   input channels of the next convolution.

6. Conv2d(32, 64, 3) therefore receives 32 learned feature
   channels and produces 64 new feature maps.

7. The architecture values 32, 64, and 128 are hyperparameters
   selected by the model designer.

8. The numerical filter weights and biases are parameters
   learned during training.

9. ReLU changes activation values without changing tensor shape.

10. Max pooling reduces spatial dimensions in this architecture.

11. Adaptive average pooling summarizes each final feature map
    into a compact value for classification.

12. Flatten converts convolutional output into a feature-vector
    representation suitable for linear layers.

13. The final Linear layer produces one raw logit per class.

14. CrossEntropyLoss will receive raw logits during training;
    softmax does not need to be placed inside the model.

15. Shape validation should be performed before model training.

16. 🔍 XAI will later investigate whether spatial feature
    representations correspond to meaningful defect regions.

17. ⚖️ RAI requires much more than a technically valid model;
    reliability must be evaluated across classes, failure cases,
    data conditions, and intended production use.